In [14]:
import pandas as pd

In [15]:
df = pd.read_csv("../../data/raw_fmri/all_fmri_10_19_2026.csv")

In [16]:
df

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Type,Acq Date,Format,Downloaded
0,I1619408,941_S_7106,MCI,F,72,sc,fMRI,Axial MB rsfMRI (Eyes Open),Original,9/09/2022,DCM,Yes
1,I1591326,941_S_7087,CN,M,67,sc,fMRI,Axial MB rsfMRI (Eyes Open),Original,6/15/2022,DCM,Yes
2,I1600185,941_S_7085,MCI,F,70,sc,fMRI,Axial MB rsfMRI (Eyes Open),Original,7/01/2022,DCM,Yes
3,I1588336,941_S_7074,CN,M,71,sc,fMRI,Axial MB rsfMRI (Eyes Open),Original,5/03/2022,DCM,Yes
4,I1553010,941_S_7051,CN,M,64,sc,fMRI,Axial MB rsfMRI (Eyes Open),Original,3/07/2022,DCM,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
9784,I569636,002_S_0413,CN,F,85,v41,fMRI,Extended Resting State fMRI,Original,6/09/2015,DCM,Yes
9785,I863058,002_S_0413,CN,F,87,init,fMRI,Axial rsfMRI (Eyes Open),Original,6/21/2017,DCM,Yes
9786,I1221056,002_S_0413,CN,F,90,y2,fMRI,Axial MB rsfMRI (Eyes Open),Original,8/27/2019,DCM,Yes
9787,I238623,002_S_0295,CN,M,90,v06,fMRI,Resting State fMRI,Original,6/02/2011,DCM,Yes


In [17]:
for a, b in df["Description"].value_counts().items():
    print(a, b)

MoCoSeries 1930
relCBF 1919
Perfusion_Weighted 1911
Axial rsfMRI (Eyes Open) 865
Resting State fMRI 767
Axial MB rsfMRI (Eyes Open) 402
Axial fcMRI (EYES OPEN) 326
Axial rsfMRI (EYES OPEN) 280
Axial fcMRI (Eyes Open) 142
Extended Resting State fMRI 134
Axial HB rsfMRI (Eyes Open) (MSV22) 98
WIP SOURCE - Axial 3D pCASL 1000 (Eyes Open) (MSV23) 92
WIP SOURCE - Axial 3D pCASL 1500 (Eyes Open) (MSV23) 90
WIP SOURCE - Axial 3D pCASL 1750 (Eyes Open) (MSV23) 89
WIP SOURCE - Axial 3D pCASL 2000 (Eyes Open) (MSV23) 89
Axial MB rsfMRI (EYES OPEN) (MSV22) 77
WIP SOURCE - Axial 3D pCASL 2500 (Eyes Open) (MSV23b) 61
Axial rsfMRI (Eyes Open) -phase P to A 48
Axial rsfMRI (Eyes Open) (MSV21) 39
WIP SOURCE - Axial 3D pCASL 1000 (Eyes Open) (MSV22) 32
WIP SOURCE - Axial 3D pCASL 1500 (Eyes Open) (MSV22) 32
WIP SOURCE - Axial 3D pCASL 1750 (Eyes Open) (MSV22) 32
WIP SOURCE - Axial 3D pCASL 2000 (Eyes Open) (MSV22) 32
WIP SOURCE - Axial 3D pCASL 2500 (Eyes Open) (MSV22) 30
Axial rsfMRI (Eyes Open) (MSV2

We need to discard a lot of these, like MoCo and ASL or DTI images.

Because your Description strings are known, you can use a single regex exclusion pattern (not_bold_pattern) that cleanly drops all 6,424 non-BOLD scans and keeps all 3,365 true BOLD rsfMRI scans:

In [18]:
# Exact pattern that matches the 6,424 non-BOLD records (MoCo, ASL/Perfusion, and DTI)
not_bold_pattern = r"MoCoSeries|relCBF|Perfusion_Weighted|pCASL|PASL|DTI"

# Keep ONLY the 3,365 true resting-state BOLD fMRI scans
clean_fmri_df = df[
    ~df["Description"].str.contains(not_bold_pattern, case=False, na=False)
].copy()

print(f"Total rows before filtering: {len(df)}")  # 9789
print(f"True BOLD rsfMRI rows kept:  {len(clean_fmri_df)}")  # 3365
print(f"Unique rsfMRI subjects:      {clean_fmri_df['Subject'].nunique()}")
print(
    f"Unique (Subject, Date) visits: {clean_fmri_df.groupby(['Subject', 'Acq Date']).ngroups}"
)

Total rows before filtering: 9789
True BOLD rsfMRI rows kept:  3365
Unique rsfMRI subjects:      1329
Unique (Subject, Date) visits: 3278


In [19]:
clean_fmri_df = clean_fmri_df.sort_values(
    by=["Subject", "Acq Date"], ascending=[True, False]
)

# All records are downloaded obviously. A
# ll are in DCM format.
# All are original type (not pre-processed or post-processed).
clean_fmri_df = clean_fmri_df.drop(columns=["Downloaded", "Format", "Type"])

In [20]:
clean_fmri_df

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date
9787,I238623,002_S_0295,CN,M,90,v06,fMRI,Resting State fMRI,6/02/2011
9788,I303069,002_S_0295,CN,M,91,v11,fMRI,Resting State fMRI,5/10/2012
9786,I1221056,002_S_0413,CN,F,90,y2,fMRI,Axial MB rsfMRI (Eyes Open),8/27/2019
9785,I863058,002_S_0413,CN,F,87,init,fMRI,Axial rsfMRI (Eyes Open),6/21/2017
9780,I240811,002_S_0413,CN,F,82,v06,fMRI,Resting State fMRI,6/16/2011
...,...,...,...,...,...,...,...,...,...
4,I1553010,941_S_7051,CN,M,64,sc,fMRI,Axial MB rsfMRI (Eyes Open),3/07/2022
3,I1588336,941_S_7074,CN,M,71,sc,fMRI,Axial MB rsfMRI (Eyes Open),5/03/2022
2,I1600185,941_S_7085,MCI,F,70,sc,fMRI,Axial MB rsfMRI (Eyes Open),7/01/2022
1,I1591326,941_S_7087,CN,M,67,sc,fMRI,Axial MB rsfMRI (Eyes Open),6/15/2022


In [21]:
duplicate_subject_dates = clean_fmri_df[
    clean_fmri_df.duplicated(subset=["Subject", "Acq Date"], keep=False)
].sort_values(["Subject", "Acq Date"])

duplicate_subject_dates

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date
9778,I322438,002_S_0685,CN,F,96,v11,fMRI,Resting State fMRI,7/27/2012
9779,I322442,002_S_0685,CN,F,96,v11,fMRI,Resting State fMRI,7/27/2012
9710,I257271,002_S_4225,CN,M,70,v02,fMRI,Resting State fMRI,9/21/2011
9711,I257275,002_S_4225,CN,M,70,v02,fMRI,Resting State fMRI,9/21/2011
9492,I265125,006_S_4150,CN,M,74,v04,fMRI,Resting State fMRI,11/07/2011
...,...,...,...,...,...,...,...,...,...
2096,I1032573,131_S_6519,CN,F,75,sc,fMRI,Axial fcMRI (EYES OPEN),7/31/2018
1706,I1185904,135_S_6446,MCI,M,63,y1,fMRI,Axial fcMRI (Eyes Open),7/11/2019
1707,I1185905,135_S_6446,MCI,M,63,y1,fMRI,Axial fcMRI (Eyes Open),7/11/2019
589,I1406560,168_S_6902,MCI,M,83,sc,fMRI,Axial rsfMRI (Eyes Open),2/04/2021


In [22]:
# Count each acquisition date once per subject, then group subjects by visit count.
visits_per_subject = (
    clean_fmri_df[["Subject", "Acq Date"]].drop_duplicates().groupby("Subject").size()
)


subjects_by_visit_count = (
    visits_per_subject.value_counts()
    .sort_index()
    .rename_axis("Number of visits")
    .rename("Number of subjects")
)

In [23]:
clean_fmri_df["Subject"].nunique()

1329

In [24]:
visits_per_subject.sum()

np.int64(3278)

In [25]:
subjects_by_visit_count

Number of visits
1     547
2     285
3     213
4     123
5      75
6      31
7      17
8      16
9      12
10      5
12      2
13      2
14      1
Name: Number of subjects, dtype: int64

In [26]:
clean_fmri_df.to_csv("../../results/dataset_creation/only_bold_fmri.csv", index=False)